# Trace-Bench Smoke Runner (Stub + Real)

This notebook validates Trace-Bench in two modes:

- **StubLLM**: deterministic, no API keys
- **Real LLM**: requires a user-provided API key (via Colab Secrets)

It also shows the standardized run artifacts produced by the CLI.

## Expected Outputs (Quick Verification)

You should see the following signals if the notebook is working correctly:

- **Stub smoke run** completes with a new `runs/<run_id>/` folder.
- `config.snapshot.yaml`, `env.json`, `results.csv`, `events.jsonl` exist in that folder.
- `results.csv` shows at least one row with `task=example:greeting_stub` and `status=trained`.
- **Real-LLM smoke** completes (if API key is set) and `results.csv` shows `status=trained`.
- `pytest -q` ends with `passed` (LLM4AD optimizer tests run only when `OPENAI_API_KEY` is set).

In [ ]:
# Mount Drive (optional) + compute persistent runs_dir
from datetime import date
from pathlib import Path
import os

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass


def bench_dir(project="bench", sub="trace_bench", local="/content/bench"):
    drive = Path("/content/drive/MyDrive")
    root = drive if drive.is_dir() else Path(local)
    out = root / project / date.today().isoformat() / sub
    out.mkdir(parents=True, exist_ok=True)
    return str(out)

RUNS_DIR = bench_dir()
os.environ["RUNS_DIR"] = RUNS_DIR
print("Runs dir:", RUNS_DIR)

In [ ]:
# Clone repos side-by-side (Trace-Bench + OpenTrace)
!git clone --depth 1 --branch runner-foundation https://github.com/guru-code-expert/Trace-Bench.git
!git clone --depth 1 --branch experimental https://github.com/guru-code-expert/OpenTrace.git

%cd Trace-Bench

# System + Python deps
!apt-get update -y && apt-get install -y graphviz
!python -m pip install -U pip
!python -m pip install pyyaml pytest numpy matplotlib graphviz litellm==1.75.0

In [ ]:
# Optional: list tasks (external bench discovery)
!python -m trace_bench list-tasks --root LLM4AD/benchmark_tasks | head -n 30

In [ ]:
%%bash
cd /content/Trace-Bench

# Stub smoke (internal example task for deterministic output)
PYTHONPATH=/content/OpenTrace:$PYTHONPATH python -m trace_bench run --config configs/smoke.yaml --runs-dir "$RUNS_DIR"

In [ ]:
# Inspect latest run artifacts
import glob, json, pathlib, pandas as pd

latest = sorted(glob.glob(f"{RUNS_DIR}/*"))[-1]
p = pathlib.Path(latest)
print(p)

print((p / "config.snapshot.yaml").read_text()[:400])
print(json.loads((p / "env.json").read_text()).keys())

pd.read_csv(p / "results.csv").head()

In [ ]:
%%bash
cd /content/Trace-Bench

# Optional: external LLM4AD smoke (may yield low score if template fails)
cat > configs/smoke_llm4ad.yaml <<'YAML'
runs_dir: runs
mode: stub
seed: 123
tasks:
  - circle_packing
trainers:
  - PrioritySearch
YAML

PYTHONPATH=/content/OpenTrace:$PYTHONPATH python -m trace_bench run --config configs/smoke_llm4ad.yaml --runs-dir "$RUNS_DIR"

## Real LLM (requires API key)

Add `OPENAI_API_KEY` in **Colab Secrets** and run the cells below.

In [ ]:
# Load API key from Colab Secrets
from google.colab import userdata
import os

key = userdata.get("OPENAI_API_KEY")
if not key:
    raise RuntimeError("Missing OPENAI_API_KEY secret in Colab")

os.environ["OPENAI_API_KEY"] = key
os.environ["TRACE_DEFAULT_LLM_BACKEND"] = "LiteLLM"
os.environ["TRACE_LITELLM_MODEL"] = "gpt-4o-mini"

In [ ]:
%%bash
cd /content/Trace-Bench

# Real-LLM smoke (internal example task)
PYTHONPATH=/content/OpenTrace:$PYTHONPATH python -m trace_bench run --config configs/smoke_real.yaml --runs-dir "$RUNS_DIR"

In [ ]:
%%bash
cd /content/Trace-Bench

# Pytest (LLM4AD optimizer test runs only if OPENAI_API_KEY is set)
PYTHONPATH=/content/OpenTrace:$PYTHONPATH python -m pytest -q